In [1]:
import numpy as np
from rocketcea.cea_obj import CEA_Obj, add_new_fuel

# --- 1. Define Constants and Inputs ---
Pc = 300.0  # Chamber Pressure (psia)
Pe = 10.0   # Exit-plane Pressure (psia)
MR = 2.0    # Mixture Ratio (Oxidizer/Fuel)
Thrust = 2500.0  # Thrust (lbf)
Cstar_eff = 0.92 # Characteristic Velocity Efficiency
gamma_guess = 1.2 # Initial guess for gamma (ratio of specific heats) for eps estimation

# n-Dodecane (C12H26) properties (as a liquid fuel, RP-1 surrogate)
# Your enthalpies of formation are in J/kg, but CEA typically uses J/mol or cal/mol.
# We will convert J/kg to cal/mol for the add_new_fuel function.
# R_universal_cal = 1.9872 cal/(mol*K)
# Molar Mass of C12H26 is 12*12.011 + 26*1.008 = 170.334 g/mol
# Hf_C12H26_J_kg = -1.45e5 J/kg (Given)
# Hf_C12H26_J_mol = Hf_C12H26_J_kg * (170.334 / 1000)  # Convert to J/mol
# Hf_C12H26_cal_mol = Hf_C12H26_J_mol / 4.184 # Convert to cal/mol
# Hf_C12H26_cal_mol = (-1.45e5 * 170.334 / 1000) / 4.184
Hf_C12H26_cal_mol = -5907.36  # Calculated value based on your given Hf

T_LOX = 90.170 # K
T_RP1 = 298.15 # K

# --- 2. Setup Propellant and Enthalpies in CEA ---

# Define the new fuel, n-Dodecane, with its Hf and T_init
# This uses the 'add_new_fuel' function to inject a custom fuel into CEA.
add_new_fuel('C12H26', hf=Hf_C12H26_cal_mol, temp=T_RP1)

# Initialize CEA_Obj for LOX/C12H26
# The 'SI=SI' tells it to use the SI-metric CEA output format.
c = CEA_Obj(propName='LOX/C12H26', oxName='LOX', fuelName='C12H26', output='si')

# For frozen flow, we must first run an equilibrium calculation to get the
# chamber properties (Tc, MW, gamma) and the required area ratio (eps).

# --- 3. Determine Expansion Area Ratio (eps) ---
# Since you have Pe, not eps, we will use a pressure ratio (Pc/Pe) in CEA
# to find the Area Ratio (eps) and then use that eps for the final frozen calculation.
Pc_over_Pe = Pc / Pe

# Get the equilibrium area ratio at the specified pressure ratio
# The 'frozen' flag is set to 0 (equilibrium) for this initial step as frozen
# flow properties are calculated *after* equilibrium in the chamber is established.
eps_eq = c.get_eps_at_PcOvPe(Pc=Pc, MR=MR, PcOvPe=Pc_over_Pe, frozen=0)

print(f"--- 🚀 CEA Calculation Setup ---")
print(f"Propellant: LOX/n-Dodecane (C12H26 surrogate)")
print(f"Mixture Ratio (MR): {MR}")
print(f"Chamber Pressure (Pc): {Pc:.1f} psia")
print(f"Exit Pressure (Pe): {Pe:.1f} psia")
print(f"Equilibrium Area Ratio (eps) found for Pc/Pe={Pc_over_Pe:.1f}: {eps_eq:.3f}")
print(f"-----------------------------------\n")

# --- 4. Calculate Requested Frozen Flow Quantities at Pc and eps ---

# Use the 'getFrozen' function to retrieve the performance parameters
# frozenAtThroat=0 means frozen from the chamber to the exit.
# frozenAtThroat=1 means equilibrium up to the throat, then frozen to the exit.
# We will assume frozenAtThroat=1 for the most common frozen-flow CEA model (IAC - Ideal Adiabatic, Frozen from Throat).
Isp_frozen_vac, Cstar_frozen, Tcomb_frozen = c.getFrozen_IvacCstrTc(
    Pc=Pc, MR=MR, eps=eps_eq, frozenAtThroat=1
)

# Get other properties at the chamber (frozen at throat, so chamber is equilibrium)
# For all of these, use frozen=0 to get the chamber (stagnation) properties
Tcomb_eq = c.get_Tcomb(Pc=Pc, MR=MR) # Adiabatic Flame Temperature (Chamber Temp)
# MW, gamma, T, Cp, H are returned at chamber, throat, and exit
ch_t_ex_MW, ch_t_ex_gamma = c.get_MolWt_gamma(Pc=Pc, MR=MR, eps=eps_eq, frozen=0)
ch_t_ex_T = c.get_Temperatures(Pc=Pc, MR=MR, eps=eps_eq, frozen=1) # Frozen T
ch_t_ex_Mach = c.get_MachNumber(Pc=Pc, MR=MR, eps=eps_eq, frozen=1) # Frozen Mach
ch_t_ex_vel = c.get_SonicVelocities(Pc=Pc, MR=MR, eps=eps_eq, frozen=1) # Frozen a
ch_t_ex_Isp = c.get_Isp(Pc=Pc, MR=MR, eps=eps_eq, frozen=1, Pamb=Pe) # Frozen Isp

# Extract specific chamber and exit values (SI Units are m/s, K, etc.)
# Chamber properties:
Chamber_T = Tcomb_eq # Adiabatic Flame Temperature (Chamber Temp)
Chamber_MW = ch_t_ex_MW[0]
Chamber_gamma = ch_t_ex_gamma[0]

# Exit properties (frozen flow - index 2):
Exh_Mach = ch_t_ex_Mach[2]
Exh_T = ch_t_ex_T[2]
Sonic_Vel_Exit = ch_t_ex_vel[2]

# Exhaust Velocity (Ve) and Specific Impulse (Isp)
# RocketCEA's get_Isp returns Veq/g0 which is Isp. The equivalent exhaust velocity (Veq)
# is calculated from the Isp: Veq = Isp * g0
g0_metric = 9.80665 # m/s^2 (standard gravity in SI)
Isp_CEA_frozen = ch_t_ex_Isp[2] # Isp at exit for Pe=10 psia (Isp_amb)
Veq_CEA_frozen = Isp_CEA_frozen * g0_metric # Equivalent Exhaust Velocity (m/s)

# Calculate theoretical C* (Cstar) and correct it for the required delivered C*
Cstar_th = c.get_Cstar(Pc=Pc, MR=MR) # Cstar (m/s)

# The required theoretical Isp to meet the 2500 lbf Thrust requirement
# Thrust = m_dot * Isp * g0
# m_dot = Thrust / (Isp_delivered * g0)
# Isp_delivered = Isp_CEA_frozen * Cstar_eff
Isp_delivered = Isp_CEA_frozen * Cstar_eff
m_dot = (Thrust / 1.0) / Isp_delivered # For US-units, Thrust[lbf] = m_dot[lbm/s] * Isp[s] * 1.0

# --- 5. Print Results in a table format ---
print("--- 🔬 CEA Frozen Flow Results (LOX/n-Dodecane) ---")
print(f"Area Ratio (eps): {eps_eq:.3f}")
print(f"C* Theoretical: {Cstar_th:.2f} m/s")
print(f"C* Delivered (Theoretical * {Cstar_eff:.2f}): {Cstar_th * Cstar_eff:.2f} m/s")
print(f"Required Mass Flow Rate (m_dot): {m_dot:.3f} lbm/s")

print("\n--- 📊 Table for Comparison (CEA Frozen Flow) ---")
print("{:<35} {:>10}".format("Quantity", "Value (SI Units)"))
print("-" * 46)
print("{:<35} {:>10.2f}".format("Adiabatic Flame Temperature (K) - Chamber", Chamber_T))
print("{:<35} {:>10.3f}".format("Chamber Molecular Weight (kg/kmol)", Chamber_MW))
print("{:<35} {:>10.4f}".format("Chamber Specific Heat Ratio (gamma)", Chamber_gamma))
print("{:<35} {:>10.4f}".format("Exhaust Mach Number - Frozen Flow", Exh_Mach))
print("{:<35} {:>10.2f}".format("Exhaust Temperature (K) - Frozen Flow", Exh_T))
print("{:<35} {:>10.2f}".format("Exhaust Velocity (m/s) - Equivalent", Veq_CEA_frozen))
print("{:<35} {:>10.2f}".format("Specific Impulse (s) - Frozen Flow (Isp_amb)", Isp_CEA_frozen))

TypeError: add_new_fuel() got an unexpected keyword argument 'hf'